Ingesting data from 9 different csvs to 9 delta tables.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

orders_schema = StructType([
    StructField('order_id',StringType(),True),
    StructField('customer_id',StringType(),True),
    StructField('order_status',StringType(),True),
    StructField('order_purchase_timestamp',TimestampType(),True),StructField('order_approved_at',TimestampType(),True),
    StructField('order_delivered_carrier_date',TimestampType(),True),StructField('order_delivered_customer_date',TimestampType(),True),
    StructField('order_estimated_delivery_date',TimestampType(),True)
])
customer_schema = StructType([
    StructField('customer_id',StringType(),True),
    StructField('customer_unique_id',StringType(),True),
    StructField('customer_zipcode_prefix',IntegerType(),True),
    StructField('customer_city',StringType(),True),
    StructField('customer_state',StringType(),True),
])
order_items_schema = StructType([
    StructField('order_id',StringType(),True),
    StructField('order_item_id',IntegerType(),True),
    StructField('product_id',StringType(),True),
    StructField('seller_id',StringType(),True),
    StructField('shipping_limit_date',TimestampType(),True),
    StructField('price',DoubleType(),True),
    StructField('freight_value',DoubleType(),True),
])
products_schema = StructType([
    StructField('product_id',StringType(),True),
    StructField('product_category_name',StringType(),True),
    StructField('product_name_length',IntegerType(),True),
    StructField('product_description_length',IntegerType(),True),
    StructField('product_photos_qty',IntegerType(),True),
    StructField('product_weight_gm',DoubleType(),True),
    StructField('product_length_cm',DoubleType(),True),
    StructField('product_height_cm',DoubleType(),True),
    StructField('product_width_cm',DoubleType(),True),
])
product_category_schema = StructType([
    StructField('product_category_name',StringType(),True),
    StructField('product_category_name_english',StringType(),True),
])
seller_schema = StructType([
    StructField('seller_id',StringType(),True),
    StructField('seller_zipcode_prefix',IntegerType(),True),
    StructField('seller_city',StringType(),True),
    StructField('seller_state',StringType(),True),
])
geolocation_schema = StructType([
    StructField('geolocation_zipcode_prefix',IntegerType(),True),
    StructField('geolocation_lat',DoubleType(),True),
    StructField('geolocation_lng',DoubleType(),True),
    StructField('geolocation_city',StringType(),True),
    StructField('geolocation_state',StringType(),True),
])
payment_schema = StructType([
    StructField('order_id',StringType(),True),
    StructField('payment_sequential',IntegerType(),True),
    StructField('payment_type',StringType(),True),
    StructField('payment_installments',IntegerType(),True),
    StructField('payment_value',DoubleType(),True),
])
review_schema = StructType([
    StructField('review_id',StringType(),True),
    StructField('order_id',StringType(),True),
    StructField('review_score',IntegerType(),True),
    StructField('review_comment_title',StringType(),True),
    StructField('review_comment_message',StringType(),True),
    StructField('review_creation_date',TimestampType(),True),StructField('review_answer_timestamp',TimestampType(),True)
])

schemas = {
    "customers.csv": customer_schema,
    "order_items.csv": order_items_schema,
    "orders.csv": orders_schema,
    "products.csv": products_schema,
    "product_category_name_translation.csv": product_category_schema,
    "sellers.csv": seller_schema,
    "geolocation.csv": geolocation_schema,
    "order_payments.csv": payment_schema,
    "order_reviews.csv": review_schema
}

In [0]:
import os

base_path =  '/Volumes/olist_ecommerce/default/raw_data/'

for file in os.listdir(base_path):
    #print(file)
    if file.endswith('.csv'):
        sch = schemas[file]
        table_name = file.replace(".csv","")
        if file == 'order_reviews.csv':
            df = spark.read.csv(base_path+file, header=True, schema=sch,quote='"',escape='"', multiLine= True)
        else:
            df = spark.read.csv(base_path+file, header=True, schema=sch)
        df.write.format('delta').mode("overwrite").saveAsTable(f"bronze_{table_name}")
    print(f"Table {table_name} created.")